# Шаг 5. Свой «Инстаграм»: фильтры на веб-камере

Видео — это просто **последовательность кадров**, а каждый кадр — такая же картинка-массив, как фото.
Значит, любой фильтр для фото работает и на видео: достаточно применять его к каждому кадру в цикле.

> ⚠️ Нужна веб-камера. Ноутбук открывает **отдельное окно OpenCV** (может оказаться позади браузера).
> Пока окно открыто, ячейка выполняется (`[*]`). Клавиши нажимайте, когда **активно окно с видео**.
> В Google Colab и на удалённом сервере не работает: там откройте `05_webcam_filters_colab.ipynb`.

**Управление:** клавиши **1–8** переключают фильтр, **s** — сохранить кадр, **q** — выход.

**Структура ноутбука (алгоритм):**
1. Импортируем OpenCV и помощник `label`.
2. Описываем фильтры как функции «кадр → кадр»: серый, карандашный рисунок, мультфильм и др.
3. Собираем словарь `FILTERS`: код клавиши → (название, функция).
4. Открываем камеру.
5. В цикле: читаем кадр → применяем текущий фильтр → подписываем → показываем → обрабатываем клавишу.
6. Освобождаем камеру и закрываем окна.

## 1. Импорт

Из `common` нужен только `label` — он рисует подпись с названием фильтра и подсказкой по клавишам.

In [ ]:
import cv2

from common import label

## 2. Фильтры-функции

Каждый фильтр — функция, которая принимает кадр `f` и возвращает обработанный кадр.

**`gray`** — перевод в оттенки серого.

**`sketch` — карандашный рисунок** (приём «осветление основы», *color dodge*):
1. переводим кадр в серый `g`;
2. делаем негатив `255 - g` и сильно размываем его (окно 21 × 21);
3. делим серое на «негатив размытого негатива»: `g / (255 - blurred) · 256`.

На ровных участках числитель и знаменатель почти равны → результат ≈ 255 (белая бумага).
На границах, где яркость резко меняется, размытая версия отличается от исходной → остаются тёмные «штрихи».

**`cartoon` — мультфильм**:
1. `bilateralFilter` сильно сглаживает цвета, но сохраняет границы — получаются ровные «заливки»;
2. `medianBlur` убирает мелкий шум с серой версии, а `adaptiveThreshold` превращает её в чёрно-белую маску:
   для каждого пикселя порог считается **по его окрестности 9 × 9** (среднее минус 2), поэтому линии находятся
   и в тёмных, и в светлых местах кадра. Тёмные линии контуров — 0, остальное — 255;
3. `bitwise_and(colors, colors, mask=edges)` оставляет цвет только там, где маска белая, — контуры становятся чёрными.

In [ ]:
def gray(f):
    return cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)


def sketch(f):
    """Карандашный рисунок: серое делим на размытый негатив."""
    g = gray(f)
    blurred = cv2.GaussianBlur(255 - g, (21, 21), 0)
    return cv2.divide(g, 255 - blurred, scale=256)


def cartoon(f):
    """Мультфильм: гладкие цвета + чёрные контуры."""
    colors = cv2.bilateralFilter(f, 9, 150, 150)
    edges = cv2.adaptiveThreshold(cv2.medianBlur(gray(f), 7), 255,
                                  cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 9, 2)
    return cv2.bitwise_and(colors, colors, mask=edges)

## 3. Словарь фильтров

Ключ — **код клавиши**: `ord("1")` возвращает числовой код символа `"1"` (49). Именно такие коды отдаёт `cv2.waitKey`.
Значение — пара `(название для подписи, функция)`.

Простые фильтры записаны как `lambda` — короткие безымянные функции в одну строку:
- `mirror` — `cv2.flip(f, 1)` отражает кадр по горизонтали (как зеркало);
- `smooth` — сильное размытие Гаусса;
- `edges` — границы Canny.

Благодаря словарю добавить новый фильтр — это одна строка, основной цикл менять не нужно.

In [ ]:
FILTERS = {
    ord("1"): ("original", lambda f: f),
    ord("2"): ("grayscale", gray),
    ord("3"): ("negative", lambda f: 255 - f),
    ord("4"): ("smooth", lambda f: cv2.GaussianBlur(f, (21, 21), 0)),
    ord("5"): ("edges", lambda f: cv2.Canny(gray(f), 60, 120)),
    ord("6"): ("mirror", lambda f: cv2.flip(f, 1)),
    ord("7"): ("sketch", sketch),
    ord("8"): ("cartoon", cartoon),
}

## 4. Открываем камеру и запускаем цикл

`cv2.VideoCapture(0)` подключается к камере номер 0 (обычно встроенная). Если камер несколько, попробуйте `1`, `2`.

Цикл обработки:
1. `cap.read()` возвращает `ok` (удалось ли прочитать кадр) и сам кадр `frame`. Если камера отключилась — выходим.
2. `apply(frame)` — применяем текущий фильтр, `label` подписывает результат.
3. `cv2.imshow` показывает кадр, `cv2.waitKey(1)` ждёт клавишу всего 1 мс, чтобы видео шло плавно.
4. Клавиша `q` — выход; клавиша из словаря — сменить фильтр; `s` — сохранить кадр в файл
   `filter_01.jpg`, `filter_02.jpg`, … (`{shot:02d}` — номер с ведущим нулём).

После цикла **обязательно** `cap.release()` — иначе камера останется занятой и следующий запуск её не откроет.

In [ ]:
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise SystemExit("Камера не открылась")

name, apply = FILTERS[ord("1")]
shot = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    picture = label(apply(frame), f"{name}   [1-8] filter  [s] save  [q] quit")
    cv2.imshow("filters", picture)

    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
        break
    if key in FILTERS:
        name, apply = FILTERS[key]
    if key == ord("s"):
        shot += 1
        cv2.imwrite(f"filter_{shot:02d}.jpg", picture)

cap.release()
cv2.destroyAllWindows()

## Попробуйте сами

1. Добавьте фильтр `9` — «сепия» или «ночное видение» (только зелёный канал: `f * [0, 1, 0]` с приведением к `uint8`).
2. Сделайте фильтр, который совмещает `mirror` и `cartoon`.
3. Выведите на кадр текущий FPS (число кадров в секунду). Подсказка: `time.perf_counter()`.
4. Почему `sketch` и `edges` возвращают серую картинку, а `label` всё равно рисует подпись? Посмотрите `to_bgr` в `common.py`.